# Exp 6b: post-training quantization of the unlearned model

Trains the Original, 5 Retrains and the fine-tuned (unlearned) model on fresh seeds 40-44 and saves the FP32 checkpoints. Then quantizes copies of them: INT8/4/3, per-channel/per-tensor, recomputed or frozen scales, plus a matched-noise control.

Main question: after quantization, is the fine-tuned model still as close to the (quantized) retrained model as in FP32?

CPU, about 30 min. On Kaggle use Save Version -> Save & Run All (Commit) so the output files are kept.

In [ ]:
# 0. Install (Colab and Kaggle already have PyTorch)
!pip install -q torch_geometric

## 1. Configuration

In [ ]:
import os, json, time, random, shutil, datetime, itertools
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torch_geometric
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T
from torch_geometric.utils import subgraph
from torch_geometric.nn import GCNConv
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from sklearn.metrics import roc_auc_score
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

SMOKE_TEST = False          # True = tiny crash test (results are NOT meaningful)
SAVE_TO_DRIVE = False       # Colab only: True saves results in Google Drive

if SAVE_TO_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/exp6_quantization'
else:
    OUT_DIR = 'exp6_quantization' if not SMOKE_TEST else 'exp6_quant_smoke_test'
for sub in ['fp32', 'quant', 'checkpoints']: os.makedirs(f'{OUT_DIR}/{sub}', exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CFG = dict(
    datasets=['CiteSeer', 'Cora'], primary_dataset='CiteSeer',
    hidden=128, dropout=0.5, lr_train=0.01, weight_decay=5e-4, epochs=200,        # frozen since Experiment 5
    train_frac=0.10, delete_frac=0.10,
    ft_lr=0.005, doses={'CiteSeer': [100], 'Cora': [100, 200]},                    # frozen FP32 doses (not re-tuned)
    n_retrain=5, seeds=[40, 41, 42, 43, 44],                                       # fresh seeds
    bits=[8, 4, 3], schemes=['per_channel', 'per_tensor'], scale_modes=['recomputed', 'frozen_original'],
    noise_draws=3, min_seeds_agree=4, utility_tolerance=0.02, collapse_drop=0.05,
    primary_config=dict(bits=4, scheme='per_channel', scale_mode='recomputed'),
    earlier_bands={'CiteSeer': {'exp6_cpu_reference': 0.0172, 'exp6_kaggle_gpu': 0.0126},
                   'Cora': {'exp6_cpu_reference': 0.0065, 'exp6_kaggle_gpu': 0.0235}},
)
if SMOKE_TEST:
    CFG.update(epochs=20, seeds=[40, 41], n_retrain=3, noise_draws=1, min_seeds_agree=1)
    print('SMOKE TEST: tiny settings, results are not meaningful.')

def log(msg):
    line = f'[{datetime.datetime.now():%H:%M:%S}] {msg}'
    print(line, flush=True)
    with open(f'{OUT_DIR}/progress.log', 'a') as f: f.write(line + '\n')

ENV = dict(device=str(DEVICE), gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           torch=torch.__version__, torch_geometric=torch_geometric.__version__, smoke_test=SMOKE_TEST,
           started=f'{datetime.datetime.now():%Y-%m-%d %H:%M:%S}')
json.dump(CFG, open(f'{OUT_DIR}/config.json', 'w'), indent=2); json.dump(ENV, open(f'{OUT_DIR}/environment.json', 'w'), indent=2)
log(f'Device: {DEVICE} | results folder: {OUT_DIR}')

## 2. Pre-registration (written and saved before any result of this stage)

In [ ]:
PREREG = f'''# Experiment 6, quantization stage: pre-registration
Saved: {datetime.datetime.now():%Y-%m-%d %H:%M:%S}, before any result of this stage. Device: {ENV['device']} ({ENV['gpu']}).

## History carried into this stage (not changed by anything below)
FP32 stage, selection seeds 20-24, evaluation seeds 30-34, band = 2 x RMS(Retrain-A - Retrain-B), one pair per seed.
* CiteSeer: both runs froze lr 0.005 / 100 steps and both passed every criterion.
  CPU reference: improvement 0.238 (95% CI 0.222-0.254), 5/5 seeds, distance 0.008 vs band 0.0172.
  Kaggle GPU:    improvement 0.241 (95% CI 0.226-0.257), 5/5 seeds, distance 0.0087 vs band 0.0126.
* Cora: borderline and not stable. CPU reference froze 200 steps, band 0.0065, distance 0.0066: NOT SUCCESSFUL.
  Kaggle GPU froze 100 steps, band 0.0235, distance 0.0066: SUCCESS. The distances agree; only the band differs (3.6x),
  within what 5 pairs allow by chance (95% range of the RMS ratio about 3.9x). Both results stay on record.

## Refinement introduced AFTER those two runs (declared as such)
The FP32 acceptance band is now estimated from {CFG['n_retrain']} independently initialised Retrain models per seed:
band_FP32 = 2 x sqrt(mean over seeds and over all Retrain pairs i<j of (AUC_i - AUC_j)^2), on the seeds of this stage.
It is computed from Retrain models only. The earlier bands are reported as a sensitivity check for both datasets.
The pairwise differences within a seed share models and are correlated, so the band is a descriptive tolerance for
Retrain-to-Retrain variation, not an independent-sample confidence bound (10 pairs per seed are not 10 independent experiments).

## Frozen settings
Datasets CiteSeer (primary) and Cora (secondary); 2-layer GCN, hidden {CFG['hidden']}, dropout {CFG['dropout']}, Adam lr {CFG['lr_train']},
weight decay {CFG['weight_decay']}, {CFG['epochs']} epochs; {int(100*CFG['train_frac'])} % training nodes; {int(100*CFG['delete_frac'])} % of them deleted.
Fine-tuning unlearning: lr {CFG['ft_lr']}, doses {CFG['doses']} (optimizer steps), retained labels only, reduced graph.
These doses are frozen and are not changed in response to any quantization result.
Seeds {CFG['seeds']} (fresh; never used before). Each seed draws its own split and initialisations.

## Models per seed (FP32, trained once and saved as checkpoints; quantization only ever uses copies of the saved weights)
Original; Retrain-1..{CFG['n_retrain']} (deleted labels and their edges removed; different initialisations; Retrain-1 is the reference);
FT at each frozen dose (started from the Original); drift control: identical fine-tuning on full data and full graph.

## Quantization (simulated, weight-only, symmetric round-to-nearest; biases stay FP32)
Bits {CFG['bits']} (levels -qmax..qmax, qmax = 2^(b-1)-1); schemes per-channel (one scale per output row) and per-tensor;
scale modes: recomputed (each model uses its own max|w|) and frozen_original (models derived from the Original, i.e. the
Original, FT and drift control, reuse the Original's scales; values outside the range are clipped; the clipped share is
reported). Retrain models are never derived from the Original, so they always use their own (recomputed) scales: they are the
reference a deployer would obtain by retraining and quantizing from scratch. Every model is quantized under every configuration.
Matched-noise control: for Original, Retrain-1 and FT, the model's own quantization error is randomly permuted within each
output row (per-channel) or within the tensor (per-tensor) and added to the FP32 weights: same error distribution and size,
no alignment with the rounding grid. {CFG['noise_draws']} draws, averaged per seed.

## Measurements (identical inference conditions for every model)
Membership: all models queried on the same full graph; score = probability of the true label; members = deleted nodes D;
non-members = held-out H_eval. AUC, advantage 2|AUC-0.5|. Calibrated attack TPR(D)-FPR(H_eval) with the threshold chosen on
retained training nodes vs H_cal (a) recalibrated on the quantized model and (b) transferred from the same model in FP32.
Mean confidence on D and on H_eval; AUC of retained training nodes vs H_eval (general memorisation signal).
Utility: held-out accuracy on the reduced graph. Accuracy on D.
Update survival (FT and drift): ||Q(W_FT)-Q(W_O)|| / ||W_FT-W_O||, cosine between the two updates, share of weights where the
quantized update is exactly zero, median |W_FT-W_O| / bin width and share above half a bin, projection of the rounding error
Q(W_FT)-W_FT onto the direction W_O-W_FT (pull toward the Original).
Node-level recovery (deleted-node specificity): on D, projection of conf_Q(FT)-conf_FT onto conf_O-conf_FT (1 = fully back to the
Original's confidences on deleted nodes); the same on H_eval as a control. Reported alongside: the raw mean confidence shifts
conf_Q(FT)-conf_FT on D and on H_eval, the raw target gaps conf_O-conf_FT on D and on H_eval, and the raw difference
(shift on D) - (shift on H_eval). A normalised recovery is flagged unstable on a seed when the RMS of its target gap
conf_O-conf_FT over that node group is below 0.01.

## Analysis per dataset, dose and configuration (paired over seeds; 95 % t-intervals; k/5 seed counts)
gap(model) = AUC(model) - AUC(Retrain-1) at the SAME precision; dist = |gap|.
band_Q = same formula as band_FP32 using the quantized Retrains; effective band = max(band_FP32, band_Q).
Increase in distance: d_dist = dist_Q(FT) - dist_FP32(FT). Reversal fraction rho = (gap_Q(FT) - gap_FP32(FT)) / (gap_FP32(O) - gap_FP32(FT)).
Excess over matched noise = d_dist - (dist_noise(FT) - dist_FP32(FT)).

## FP32 gate (seeds of this stage, before quantization)
Per dataset and dose: paired improvement dist(O) - dist(FT) with CI above 0 and > 0 on >= {CFG['min_seeds_agree']} seeds; mean dist(FT) <= band_FP32;
utility(FT) >= utility(O) - {CFG['utility_tolerance']}; the Original fails the band (mean dist(O) > band_FP32 and on >= {CFG['min_seeds_agree']} seeds).
Only combinations that pass are analysed as confirmatory. Others are not quantized (smoke tests excepted).

## Verdict per configuration (in this order)
1. Not interpretable, model degraded: mean held-out accuracy of the quantized Original, Retrain-1 or FT drops by more than
   {CFG['collapse_drop']} vs the same model in FP32.
2. Not interpretable, attack signal lost: the quantized Original no longer fails the effective band (mean).
3. REVERSAL (measured attack): mean dist_Q(FT) > effective band; d_dist CI above 0 and > 0 on >= {CFG['min_seeds_agree']} seeds; excess over matched
   noise CI above 0 and > 0 on >= {CFG['min_seeds_agree']} seeds. Sub-label (does not change the verdict): "deleted-node-specific shift" if node
   recovery on D exceeds node recovery on H_eval (paired CI above 0) AND the raw shift difference D - H_eval has CI above 0 AND no
   seed is flagged unstable; "specificity not assessable" if any seed is flagged unstable; otherwise "non-specific shift".
   A deleted-node-specific shift is supporting evidence for recovered deleted-node information, not proof of it.
4. PRESERVED: mean dist_Q(FT) <= effective band and dist_Q(FT) <= effective band on >= {CFG['min_seeds_agree']} seeds.
5. Otherwise INCONCLUSIVE.
Frozen-scale configurations combine rounding with clipping (FT weights can outgrow the Original's range). If the mean clipped
share of FT weights exceeds 1 %, that configuration is labelled "clipping-dominated" and is not read as a rounding effect.
Primary test: {CFG['primary_dataset']}, INT{CFG['primary_config']['bits']}, {CFG['primary_config']['scheme']}, {CFG['primary_config']['scale_mode']} scales. All other
configurations, Cora and all doses are secondary and are all reported, including failures.

## If no dataset/dose passes the FP32 gate
No quantization is run; the outcome NO_FP32_GATE_PASSED is saved with the FP32 results and exported. This is a valid result.

## Scope of the conclusion
Conclusions are about this measured membership attack (and its calibrated variants), not formal privacy guarantees. A change in
the attack is only read as recovery of deleted-node information if it passes the matched-noise and node-level checks; a change
in the transferred-threshold attack without a change in AUC is reported as a calibration shift.
'''
path = f'{OUT_DIR}/exp6q_preregistration.md'
if os.path.exists(path):
    log('Pre-registration already saved earlier: keeping the original file unchanged.')
else:
    open(path, 'w').write(PREREG); log('Pre-registration saved.')
print(open(path).read())

## 3. Model
Same sparse-input GCN as the FP32 stage: each layer computes Â·(X Wᵀ) + b, exactly like `GCNConv` (checked below).

In [ ]:
def norm_adj(edge_index, n):
    ei, w = gcn_norm(edge_index, None, n, add_self_loops=True)
    return torch.sparse_coo_tensor(torch.stack([ei[1], ei[0]]), w, (n, n)).coalesce().to(DEVICE)

class Layer(nn.Module):
    def __init__(self, i, o):
        super().__init__(); self.lin = nn.Linear(i, o, bias=False); self.bias = nn.Parameter(torch.zeros(o))
        nn.init.xavier_uniform_(self.lin.weight)
    def forward(self, x, A):
        xw = torch.sparse.mm(x, self.lin.weight.t()) if x.is_sparse else x @ self.lin.weight.t()
        return torch.sparse.mm(A, xw) + self.bias

class GCN(nn.Module):
    def __init__(self, i, h, o, p):
        super().__init__(); self.c1, self.c2, self.p = Layer(i, h), Layer(h, o), p
    def forward(self, xs, A):
        if self.training and self.p > 0:
            xs = torch.sparse_coo_tensor(xs.indices(), F.dropout(xs.values(), self.p, True), xs.shape)
        h = F.relu(self.c1(xs, A)); h = F.dropout(h, self.p, self.training)
        return self.c2(h, A)

# parameter order: c1.bias, c1.lin.weight (2-D), c2.bias, c2.lin.weight (2-D). Only the 2-D weights are quantized
# (the code selects them by shape, never by position).
_d = Planetoid('data', 'Cora', transform=T.NormalizeFeatures())[0]
_m = GCN(_d.num_features, 128, 7, 0.5).to(DEVICE).eval()
assert [p.dim() for p in _m.parameters()] == [1, 2, 1, 2]
_r1, _r2 = GCNConv(_d.num_features, 128).to(DEVICE), GCNConv(128, 7).to(DEVICE)
with torch.no_grad():
    _m.c1.bias.add_(0.1)
    _r1.lin.weight.copy_(_m.c1.lin.weight); _r1.bias.copy_(_m.c1.bias); _r2.lin.weight.copy_(_m.c2.lin.weight); _r2.bias.copy_(_m.c2.bias)
    diff = (_m(_d.x.to_sparse().to(DEVICE), norm_adj(_d.edge_index, _d.num_nodes))
            - _r2(F.relu(_r1(_d.x.to(DEVICE), _d.edge_index.to(DEVICE))), _d.edge_index.to(DEVICE))).abs().max().item()
log(f'Equivalence check vs PyG GCNConv: max |difference| = {diff:.2e}')
assert diff < 1e-4, 'implementation does not match GCNConv'

## 4. Quantizer, matched noise and measurements
`quantize` is simulated (fake) quantization: weights are rounded to the integer grid and immediately mapped back to floats.
The checks at the end of this cell verify it on known numbers before anything is run.

In [ ]:
def quantize(W, bits, scheme, scale_src=None):
    # W: list of parameters; returns (quantized copy, list of per-element scales or None, share of clipped weights)
    qmax = 2 ** (bits - 1) - 1; out, scales, clipped, total = [], [], 0, 0
    src = W if scale_src is None else scale_src
    for w, s0 in zip(W, src):
        if w.dim() != 2: out.append(w.clone()); scales.append(None); continue
        s = (s0.abs().amax(1, keepdim=True) if scheme == 'per_channel' else s0.abs().max()) / qmax
        s = torch.clamp(s, min=1e-12).expand_as(w)
        q = torch.round(w / s); clipped += int((q.abs() > qmax).sum()); total += q.numel()
        out.append(torch.clamp(q, -qmax, qmax) * s); scales.append(s)
    return out, scales, clipped / total

def matched_noise(W, Wq, scheme, gen):
    # adds the model's own quantization error, randomly permuted within each row (per-channel) or the tensor (per-tensor)
    out = []
    for w, wq in zip(W, Wq):
        if w.dim() != 2: out.append(w.clone()); continue
        e = (wq - w).cpu()
        if scheme == 'per_channel':
            idx = torch.argsort(torch.rand(e.shape, generator=gen), dim=1); e = torch.gather(e, 1, idx)
        else:
            e = e.flatten()[torch.randperm(e.numel(), generator=gen)].view_as(e)
        out.append(w + e.to(w.device))
    return out

def flat2d(W): return torch.cat([w.flatten() for w in W if w.dim() == 2])
def cos(a, b): return float((a @ b) / (a.norm() * b.norm() + 1e-30))

# --- self-checks on known numbers ---
_w = [torch.tensor([[0.70, -0.35, 0.10], [0.07, 0.02, -0.01]]), torch.zeros(2)]
_q, _s, _c = quantize(_w, 4, 'per_channel')       # row scales 0.1 and 0.01
assert torch.allclose(_q[0], torch.tensor([[0.7, -0.4, 0.1], [0.07, 0.02, -0.01]]), atol=1e-6), _q[0]
_q, _s, _c = quantize(_w, 4, 'per_tensor')        # one scale 0.1
assert torch.allclose(_q[0], torch.tensor([[0.7, -0.4, 0.1], [0.1, 0.0, 0.0]]), atol=1e-6), _q[0]
_q, _s, _c = quantize([_w[0] * 2, _w[1]], 4, 'per_channel', scale_src=_w)   # frozen scales -> clipping
assert abs(_c - 2 / 6) < 1e-9 and torch.allclose(_q[0][0], torch.tensor([0.7, -0.7, 0.2]), atol=1e-6)
_q, _s, _c = quantize(_w, 8, 'per_channel'); assert (_q[0] - _w[0]).abs().max() <= 0.7 / 127 / 2 + 1e-7
_n = matched_noise(_w, quantize(_w, 4, 'per_channel')[0], 'per_channel', torch.Generator().manual_seed(0))
_e1, _e2 = (quantize(_w, 4, 'per_channel')[0][0] - _w[0]), (_n[0] - _w[0])
assert torch.allclose(_e1.sort(1).values, _e2.sort(1).values, atol=1e-7)     # same errors per row, permuted
log('Quantizer and matched-noise self-checks passed.')

_DATA, _PROBE = {}, {}
def get_data(name):
    if name not in _DATA:
        d = Planetoid('data', name, transform=T.NormalizeFeatures())[0]
        _DATA[name] = dict(d=d, xs=d.x.to_sparse().to(DEVICE), y=d.y.to(DEVICE), y_np=d.y.numpy(),
                           A_full=norm_adj(d.edge_index, d.num_nodes), C=int(d.y.max()) + 1)
    return _DATA[name]

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def make_split(ds, seed):          # identical to the FP32 stage
    G = get_data(ds); d = G['d']; N = d.num_nodes
    rng = np.random.default_rng(seed)
    perm = rng.permutation(N); n_tr = int(CFG['train_frac'] * N)
    tr, held = np.sort(perm[:n_tr]), np.sort(perm[n_tr:])
    D = np.sort(rng.choice(tr, int(CFG['delete_frac'] * len(tr)), replace=False)); retained = np.setdiff1d(tr, D)
    hp = rng.permutation(held); H_cal, H_eval = hp[: len(hp) // 2], hp[len(hp) // 2:]
    R_cal = rng.choice(retained, size=min(len(retained), len(H_cal)), replace=False)
    return dict(train=tr, held=held, D=D, retained=retained, H_cal=H_cal, H_eval=H_eval, R_cal=R_cal)

def make_ctx(ds, sp):
    G = get_data(ds); N = G['d'].num_nodes
    keep = torch.ones(N, dtype=torch.bool); keep[torch.from_numpy(sp['D'])] = False
    ei_red, _ = subgraph(keep, G['d'].edge_index, num_nodes=N)
    if ds not in _PROBE: _PROBE[ds] = GCN(G['d'].num_features, CFG['hidden'], G['C'], CFG['dropout']).to(DEVICE)
    return dict(ds=ds, G=G, sp=sp, A_red=norm_adj(ei_red, N), N=N)

def calibrate(cr, ch):
    # threshold maximising (TPR on retained + TNR on H_cal)/2; member if conf >= t (vectorised, first maximum)
    cand = np.unique(np.r_[cr, ch]); sr, sh = np.sort(cr), np.sort(ch)
    tpr = 1 - np.searchsorted(sr, cand, 'left') / len(sr); tnr = np.searchsorted(sh, cand, 'left') / len(sh)
    return float(cand[int(np.argmax((tpr + tnr) / 2))])

@torch.no_grad()
def measure(ctx, W, thr_fp32=None):
    G, sp, probe = ctx['G'], ctx['sp'], _PROBE[ctx['ds']]
    for p, w in zip(probe.parameters(), W): p.data = w.clone()
    probe.eval(); y = G['y_np']; N = ctx['N']
    conf = F.softmax(probe(G['xs'], G['A_full']), 1).cpu().numpy()[np.arange(N), y]
    pred_red = probe(G['xs'], ctx['A_red']).argmax(1).cpu().numpy()
    D, He, ret = sp['D'], sp['H_eval'], sp['retained']
    auc = roc_auc_score(np.r_[np.ones(len(D)), np.zeros(len(He))], np.r_[conf[D], conf[He]])
    thr = calibrate(conf[sp['R_cal']], conf[sp['H_cal']])
    t2 = thr if thr_fp32 is None else thr_fp32
    m = dict(auc=auc, adv=2 * abs(auc - 0.5), threshold=thr,
             tpr_minus_fpr_recal=(conf[D] >= thr).mean() - (conf[He] >= thr).mean(),
             tpr_minus_fpr_fp32_threshold=(conf[D] >= t2).mean() - (conf[He] >= t2).mean(),
             mean_conf_D=float(conf[D].mean()), mean_conf_Heval=float(conf[He].mean()),
             auc_retained_vs_Heval=roc_auc_score(np.r_[np.ones(len(ret)), np.zeros(len(He))], np.r_[conf[ret], conf[He]]),
             test_acc=float((pred_red[sp['held']] == y[sp['held']]).mean()), acc_deleted=float((pred_red[D] == y[D]).mean()))
    return m, conf

UNSTABLE_RMS = 0.01     # pre-registered: normalised recovery is unstable if RMS(conf_O - conf_FT) over the group < 0.01
def node_recovery(sp, conf_new, conf_ft, conf_o):
    out = {}
    for name, idx in [('D', sp['D']), ('Heval', sp['H_eval'])]:
        t, dlt = conf_o[idx] - conf_ft[idx], conf_new[idx] - conf_ft[idx]
        rms = float(np.sqrt(np.mean(t ** 2)))
        out.update({f'node_recovery_{name}': float(dlt @ t / (t @ t + 1e-30)), f'raw_shift_{name}': float(dlt.mean()),
                    f'raw_target_gap_{name}': float(t.mean()), f'target_rms_{name}': rms, f'unstable_{name}': bool(rms < UNSTABLE_RMS)})
    out['raw_shift_D_minus_Heval'] = out['raw_shift_D'] - out['raw_shift_Heval']
    return out

## 5. FP32 phase: train once, save exact checkpoints, evaluate in FP32

In [ ]:
def model_names(ds):
    return (['Original'] + [f'Retrain-{k}' for k in range(1, CFG['n_retrain'] + 1)]
            + [f'FT dose={d}' for d in CFG['doses'][ds]] + [f'Drift dose={d}' for d in CFG['doses'][ds]])

def ck_path(ds, seed): return f'{OUT_DIR}/checkpoints/{ds}_seed{seed}.pt'

def train_fp32(ds, seed):
    if os.path.exists(ck_path(ds, seed)):
        log(f'{ds} seed {seed}: FP32 checkpoint exists, not retraining'); return
    t0 = time.time(); G = get_data(ds); d, xs, Y, A_full = G['d'], G['xs'], G['y'], G['A_full']
    sp = make_split(ds, seed); ctx = make_ctx(ds, sp); A_red = ctx['A_red']; N = d.num_nodes
    m_full = torch.zeros(N, dtype=torch.bool, device=DEVICE); m_full[torch.from_numpy(sp['train'])] = True
    m_red = torch.zeros(N, dtype=torch.bool, device=DEVICE); m_red[torch.from_numpy(sp['retained'])] = True
    def train(A, mask, init_seed):
        set_seed(init_seed); m = GCN(d.num_features, CFG['hidden'], G['C'], CFG['dropout']).to(DEVICE)
        opt = torch.optim.Adam(m.parameters(), lr=CFG['lr_train'], weight_decay=CFG['weight_decay'])
        for _ in range(CFG['epochs']):
            m.train(); opt.zero_grad(); F.cross_entropy(m(xs, A)[mask], Y[mask]).backward(); opt.step()
        return [p.detach().clone() for p in m.parameters()]
    W = {'Original': train(A_full, m_full, 1000 + seed)}
    for k in range(1, CFG['n_retrain'] + 1):               # Retrain-1/-2 use the same init seeds as Retrain-A/-B before
        W[f'Retrain-{k}'] = train(A_red, m_red, 2000 + 1000 * (k - 1) + seed)
    for kind, A, mask in [('FT', A_red, m_red), ('Drift', A_full, m_full)]:
        set_seed(5000 + seed + int(CFG['ft_lr'] * 1e4))
        m = GCN(d.num_features, CFG['hidden'], G['C'], CFG['dropout']).to(DEVICE)
        for p, w in zip(m.parameters(), W['Original']): p.data = w.clone()
        opt = torch.optim.Adam(m.parameters(), lr=CFG['ft_lr'], weight_decay=CFG['weight_decay'])
        for step in range(1, max(CFG['doses'][ds]) + 1):
            m.train(); opt.zero_grad(); F.cross_entropy(m(xs, A)[mask], Y[mask]).backward(); opt.step()
            if step in CFG['doses'][ds]: W[f'{kind} dose={step}'] = [p.detach().clone() for p in m.parameters()]
    torch.save(dict(weights={k: [w.cpu() for w in v] for k, v in W.items()}, split=sp, seed=seed, dataset=ds, env=ENV),
               ck_path(ds, seed))
    log(f'{ds} seed {seed}: trained and saved FP32 checkpoint in {time.time()-t0:.0f}s')

def load_ck(ds, seed):
    ck = torch.load(ck_path(ds, seed), weights_only=False)
    return {k: [w.to(DEVICE) for w in v] for k, v in ck['weights'].items()}, ck['split']

def eval_fp32(ds, seed):
    f = f'{OUT_DIR}/fp32/{ds}_seed{seed}.csv'
    if os.path.exists(f): return
    W, sp = load_ck(ds, seed); ctx = make_ctx(ds, sp); rows = []
    for name in model_names(ds):
        m, _ = measure(ctx, W[name])
        rows.append(dict(dataset=ds, seed=seed, model=name, update_norm=float(flat2d(W[name]).sub(flat2d(W['Original'])).norm()), **m))
    pd.DataFrame(rows).to_csv(f, index=False)

for ds in CFG['datasets']:
    for seed in CFG['seeds']:
        train_fp32(ds, seed); eval_fp32(ds, seed)
log('FP32 phase finished.')

## 6. FP32 gate (decided and saved before any quantization)

In [ ]:
def ci95(x):
    x = np.asarray(x, float); n = len(x); m = x.mean()
    if n < 2: return m, np.nan, np.nan
    h = stats.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n); return m, m - h, m + h

def retrain_band(auc_by_seed):
    # auc_by_seed: DataFrame index seed, columns Retrain-1..K
    cols = [c for c in auc_by_seed.columns if c.startswith('Retrain-')]
    sq = [(auc_by_seed[a] - auc_by_seed[b]) ** 2 for a, b in itertools.combinations(cols, 2)]
    return float(2 * np.sqrt(pd.concat(sq).mean()))

FP = pd.concat([pd.read_csv(f'{OUT_DIR}/fp32/{ds}_seed{s}.csv') for ds in CFG['datasets'] for s in CFG['seeds']])
GATE_PATH = f'{OUT_DIR}/fp32_gate.json'
if os.path.exists(GATE_PATH):
    GATE = json.load(open(GATE_PATH)); log('FP32 gate already decided earlier: NOT recomputed.')
else:
    GATE = {}
    for ds in CFG['datasets']:
        a = FP[FP.dataset == ds].pivot_table(index='seed', columns='model', values='auc')
        u = FP[FP.dataset == ds].pivot_table(index='seed', columns='model', values='test_acc')
        band = retrain_band(a); k = CFG['min_seeds_agree']
        dist_o = (a['Original'] - a['Retrain-1']).abs()
        other_r = [(a[c] - a['Retrain-1']).abs() for c in a.columns if c.startswith('Retrain-') and c != 'Retrain-1']
        GATE[ds] = dict(band_fp32=band, earlier_bands=CFG['earlier_bands'][ds], dist_original=float(dist_o.mean()),
                        retrains_within_band=f"{int(sum((r <= band).sum() for r in other_r))}/{sum(len(r) for r in other_r)}", doses={})
        for dose in CFG['doses'][ds]:
            ft = f'FT dose={dose}'; dist_ft = (a[ft] - a['Retrain-1']).abs(); imp = dist_o - dist_ft
            m, lo, hi = ci95(imp)
            crit = dict(ci_above_0=bool(lo > 0), improves_on_enough_seeds=bool((imp > 0).sum() >= k),
                        within_band=bool(dist_ft.mean() <= band),
                        utility_ok=bool(u[ft].mean() >= u['Original'].mean() - CFG['utility_tolerance']),
                        original_fails_band=bool(dist_o.mean() > band and (dist_o > band).sum() >= k))
            GATE[ds]['doses'][str(dose)] = dict(improvement=float(m), ci95=[float(lo), float(hi)], seeds_improving=f'{int((imp > 0).sum())}/{len(imp)}',
                dist_ft=float(dist_ft.mean()), utility_ft=float(u[ft].mean()), utility_original=float(u['Original'].mean()),
                within_earlier_bands={kk: bool(dist_ft.mean() <= v) for kk, v in CFG['earlier_bands'][ds].items()},
                criteria=crit, passed=all(crit.values()))
    json.dump(GATE, open(GATE_PATH, 'w'), indent=2); log('FP32 gate saved.')

for ds, g in GATE.items():
    print(f"\n=== {ds}: band_FP32 {g['band_fp32']:.4f} (earlier: {g['earlier_bands']}) | dist(Original) {g['dist_original']:.3f} | "
          f"other Retrains within band: {g['retrains_within_band']} ===")
    for dose, r in g['doses'].items():
        print(f"  dose {dose}: improvement {r['improvement']:.4f} (CI {r['ci95'][0]:.4f} to {r['ci95'][1]:.4f}), {r['seeds_improving']} seeds, "
              f"dist {r['dist_ft']:.4f}, utility {r['utility_ft']:.3f} vs {r['utility_original']:.3f}")
        print(f"  criteria {r['criteria']} => {'PASSED' if r['passed'] else 'NOT PASSED'} | within earlier bands: {r['within_earlier_bands']}")

print('\nNote: the band is a descriptive tolerance (correlated Retrain pairs), not an independent-sample confidence bound.')
TO_QUANTIZE = {ds: [int(d) for d, r in g['doses'].items() if r['passed'] or SMOKE_TEST] for ds, g in GATE.items()}
if SMOKE_TEST: print('\nSMOKE TEST: quantization runs regardless of the gate, to test the code.')
log(f'Quantized (gate passed): {TO_QUANTIZE}')

## 7. Quantization phase: quantize copies of the saved checkpoints
Every model × 3 bit-widths × 2 schemes × 2 scale modes, plus matched-noise draws for Original, Retrain-1 and FT.
One CSV per seed, written when the seed is finished; finished seeds are skipped on *Run all*.

In [ ]:
CONFIGS = list(itertools.product(CFG['bits'], CFG['schemes'], CFG['scale_modes']))

def quant_seed(ds, seed, doses):
    f = f'{OUT_DIR}/quant/{ds}_seed{seed}.csv'
    if os.path.exists(f): log(f'{ds} seed {seed}: quantization already done, skipping'); return
    t0 = time.time(); W, sp = load_ck(ds, seed); ctx = make_ctx(ds, sp)       # exact saved FP32 weights
    names = [n for n in model_names(ds) if not n.split('=')[-1].isdigit() or int(n.split('=')[-1]) in doses]
    fp = {n: measure(ctx, W[n]) for n in names}                               # (metrics, conf) in FP32
    base = dict(dataset=ds, seed=seed)
    rows = [dict(**base, model=n, bits=32, scheme='fp32', scale_mode='fp32', condition='fp32', draw=-1, clip_frac=0.0, **fp[n][0])
            for n in names]
    for bits, scheme, mode in CONFIGS:
        cfg = dict(bits=bits, scheme=scheme, scale_mode=mode)
        def src(n): return W['Original'] if (mode == 'frozen_original' and not n.startswith('Retrain')) else None
        Q = {n: quantize(W[n], bits, scheme, src(n)) for n in names}
        for n in names:
            Wq, S, clip = Q[n]; m, conf = measure(ctx, Wq, thr_fp32=fp[n][0]['threshold'])
            r = dict(**base, model=n, **cfg, condition='quantized', draw=-1, clip_frac=clip, **m)
            if n.startswith(('FT', 'Drift')):
                d_fp = flat2d(W[n]) - flat2d(W['Original']); d_q = flat2d(Wq) - flat2d(Q['Original'][0])
                s = torch.cat([x.flatten() for x in S if x is not None]); rel = d_fp.abs() / s
                err = flat2d(Wq) - flat2d(W[n]); to_o = flat2d(W['Original']) - flat2d(W[n])
                r.update(upd_norm_fp32=float(d_fp.norm()), upd_norm_ratio=float(d_q.norm() / d_fp.norm()), upd_cosine=cos(d_q, d_fp),
                         share_upd_erased=float((d_q == 0).float().mean()), med_upd_over_bin=float(rel.median()),
                         share_upd_above_half_bin=float((rel > 0.5).float().mean()),
                         rounding_pull_to_original=float(err @ to_o / (to_o @ to_o)), rounding_cos_to_original=cos(err, to_o),
                         **node_recovery(sp, conf, fp[n][1], fp['Original'][1]))
            rows.append(r)
        for n in [x for x in names if x in ('Original', 'Retrain-1') or x.startswith('FT')]:
            for draw in range(CFG['noise_draws']):
                gen = torch.Generator().manual_seed(9000 + 100 * seed + 10 * draw + bits)
                m, conf = measure(ctx, matched_noise(W[n], Q[n][0], scheme, gen), thr_fp32=fp[n][0]['threshold'])
                r = dict(**base, model=n, **cfg, condition='matched_noise', draw=draw, clip_frac=np.nan, **m)
                if n.startswith('FT'): r.update(node_recovery(sp, conf, fp[n][1], fp['Original'][1]))
                rows.append(r)
    pd.DataFrame(rows).to_csv(f, index=False)
    log(f'{ds} seed {seed}: quantization done in {time.time()-t0:.0f}s ({len(rows)} rows)')

NO_GATE = not any(TO_QUANTIZE.values())
if NO_GATE:
    json.dump(dict(outcome='NO_FP32_GATE_PASSED', gate=GATE,
                   note='No dataset/dose passed the pre-registered FP32 gate; quantization was not run. FP32 results are in fp32/.'),
              open(f'{OUT_DIR}/quantization_outcome.json', 'w'), indent=2)
    log('NO_FP32_GATE_PASSED: no quantization run. This is a valid, pre-registered negative result; continue to the export.')
for ds in CFG['datasets']:
    if not TO_QUANTIZE[ds]: log(f'{ds}: no dose passed the FP32 gate, not quantized'); continue
    for seed in CFG['seeds']: quant_seed(ds, seed, TO_QUANTIZE[ds])
log('Quantization phase finished.')

## 8. Pre-registered analysis and verdicts (all configurations reported)

In [ ]:
_qf = [f'{OUT_DIR}/quant/{ds}_seed{s}.csv' for ds in CFG['datasets'] if TO_QUANTIZE[ds] for s in CFG['seeds']]
_qf = [f for f in _qf if os.path.exists(f)]
QD = pd.concat([pd.read_csv(f) for f in _qf]) if _qf else pd.DataFrame()
k = CFG['min_seeds_agree']; summary, outcome = [], {}
if QD.empty: print('NO_FP32_GATE_PASSED: nothing to analyse. The FP32 results and the gate are saved; go to the export.')
for ds in (CFG['datasets'] if not QD.empty else []):
    for dose in TO_QUANTIZE[ds]:
        ft = f'FT dose={dose}'; x = QD[QD.dataset == ds]
        f32 = x[x.condition == 'fp32'].pivot_table(index='seed', columns='model', values='auc')
        acc32 = x[x.condition == 'fp32'].pivot_table(index='seed', columns='model', values='test_acc')
        band_fp = GATE[ds]['band_fp32']
        gap_fp_ft, gap_fp_o = f32[ft] - f32['Retrain-1'], f32['Original'] - f32['Retrain-1']; dist_fp_ft = gap_fp_ft.abs()
        for (bits, scheme, mode), g in x[x.condition != 'fp32'].groupby(['bits', 'scheme', 'scale_mode']):
            q = g[g.condition == 'quantized']; nz = g[g.condition == 'matched_noise']
            a = q.pivot_table(index='seed', columns='model', values='auc'); acc = q.pivot_table(index='seed', columns='model', values='test_acc')
            an = nz.pivot_table(index='seed', columns='model', values='auc')        # averaged over draws
            band_q = retrain_band(a); eff = max(band_fp, band_q)
            gap_q = a[ft] - a['Retrain-1']; dist_q = gap_q.abs(); dist_q_o = (a['Original'] - a['Retrain-1']).abs()
            d_dist = dist_q - dist_fp_ft; rho = (gap_q - gap_fp_ft) / (gap_fp_o - gap_fp_ft)
            excess = d_dist - ((an[ft] - an['Retrain-1']).abs() - dist_fp_ft)
            qf = q[q.model == ft].set_index('seed'); nr = qf.node_recovery_D - qf.node_recovery_Heval
            dd, ex, nrs, rh, raw = ci95(d_dist), ci95(excess), ci95(nr), ci95(rho), ci95(qf.raw_shift_D_minus_Heval)
            n_unstable = int((qf.unstable_D.astype(bool) | qf.unstable_Heval.astype(bool)).sum())
            drops = {mm: float(acc32[mm].mean() - acc[mm].mean()) for mm in ['Original', 'Retrain-1', ft]}
            acc_drop = max(drops.values())
            if acc_drop > CFG['collapse_drop']: verdict = 'NOT INTERPRETABLE: model degraded'
            elif not dist_q_o.mean() > eff: verdict = 'NOT INTERPRETABLE: attack signal lost'
            elif (dist_q.mean() > eff and dd[1] > 0 and (d_dist > 0).sum() >= k and ex[1] > 0 and (excess > 0).sum() >= k):
                verdict = 'REVERSAL, ' + ('specificity not assessable' if n_unstable else
                                          'deleted-node-specific shift' if (nrs[1] > 0 and raw[1] > 0) else 'non-specific shift')
            elif dist_q.mean() <= eff and (dist_q <= eff).sum() >= k: verdict = 'PRESERVED'
            else: verdict = 'INCONCLUSIVE'
            if mode == 'frozen_original' and q[q.model == ft].clip_frac.mean() > 0.01: verdict += ' [clipping-dominated]'
            primary = (ds == CFG['primary_dataset'] and bits == CFG['primary_config']['bits'] and scheme == CFG['primary_config']['scheme']
                       and mode == CFG['primary_config']['scale_mode'] and dose == CFG['doses'][ds][0])
            summary.append(dict(dataset=ds, dose=dose, bits=bits, scheme=scheme, scale_mode=mode, primary=primary, verdict=verdict,
                band_fp32=band_fp, band_q=band_q, effective_band=eff,
                dist_fp32_FT=dist_fp_ft.mean(), dist_q_FT=dist_q.mean(), seeds_within_band=f'{int((dist_q <= eff).sum())}/{len(dist_q)}',
                dist_q_Original=dist_q_o.mean(), increase_dist=dd[0], increase_ci_lo=dd[1], increase_ci_hi=dd[2],
                seeds_increase=f'{int((d_dist > 0).sum())}/{len(d_dist)}', reversal_fraction=rh[0], reversal_ci_lo=rh[1], reversal_ci_hi=rh[2],
                excess_over_noise=ex[0], excess_ci_lo=ex[1], excess_ci_hi=ex[2], seeds_excess=f'{int((excess > 0).sum())}/{len(excess)}',
                node_recovery_D=qf.node_recovery_D.mean(), node_recovery_Heval=qf.node_recovery_Heval.mean(), node_specific_ci_lo=nrs[1],
                raw_shift_D=qf.raw_shift_D.mean(), raw_shift_Heval=qf.raw_shift_Heval.mean(), raw_target_gap_D=qf.raw_target_gap_D.mean(),
                raw_target_gap_Heval=qf.raw_target_gap_Heval.mean(), raw_shift_D_minus_Heval=raw[0], raw_diff_ci_lo=raw[1], raw_diff_ci_hi=raw[2],
                seeds_unstable_recovery=f'{n_unstable}/{len(qf)}',
                acc_drop_max=acc_drop, acc_drop_Original=drops['Original'], acc_drop_FT=drops[ft], acc_q_FT=acc[ft].mean(), acc_fp32_FT=acc32[ft].mean(),
                tpr_fpr_recal_FT=qf.tpr_minus_fpr_recal.mean(), tpr_fpr_fp32thr_FT=qf.tpr_minus_fpr_fp32_threshold.mean(),
                upd_norm_ratio=qf.upd_norm_ratio.mean(), upd_cosine=qf.upd_cosine.mean(), share_upd_erased=qf.share_upd_erased.mean(),
                med_upd_over_bin=qf.med_upd_over_bin.mean(), rounding_pull_to_original=qf.rounding_pull_to_original.mean(),
                clip_frac_FT=qf.clip_frac.mean()))
S = pd.DataFrame(summary)
if S.empty: S = pd.DataFrame(columns=['dataset', 'dose', 'primary', 'verdict'])
S.to_csv(f'{OUT_DIR}/quantization_summary.csv', index=False)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60)
for ds in (CFG['datasets'] if not QD.empty else []):
    for dose in TO_QUANTIZE[ds]:
        s = S[(S.dataset == ds) & (S.dose == dose)]
        print(f'\n=== {ds}, FT dose {dose} ===')
        display(s[['bits', 'scheme', 'scale_mode', 'verdict', 'effective_band', 'dist_fp32_FT', 'dist_q_FT', 'seeds_within_band',
                   'increase_dist', 'increase_ci_lo', 'reversal_fraction', 'excess_over_noise', 'node_recovery_D', 'node_recovery_Heval', 'raw_shift_D', 'raw_shift_Heval', 'seeds_unstable_recovery',
                   'acc_drop_max', 'clip_frac_FT', 'upd_cosine', 'upd_norm_ratio', 'med_upd_over_bin']].round(4))
P = S[S.primary]
if len(P):
    p = P.iloc[0]
    print(f"\nPRIMARY ({p.dataset}, INT{p.bits} {p.scheme} {p.scale_mode}, dose {p.dose}): {p.verdict} | dist {p.dist_q_FT:.4f} vs "
          f"effective band {p.effective_band:.4f} | increase {p.increase_dist:.4f} (CI {p.increase_ci_lo:.4f} to {p.increase_ci_hi:.4f}) | "
          f"reversal fraction {p.reversal_fraction:.3f} (CI {p.reversal_ci_lo:.3f} to {p.reversal_ci_hi:.3f})")
if not QD.empty:
    outcome = dict(outcome='ANALYSED', primary=P.to_dict('records'),
                   verdict_counts=S.groupby(['dataset', 'dose', 'verdict']).size().reset_index(name='n').to_dict('records'))
    json.dump(outcome, open(f'{OUT_DIR}/quantization_outcome.json', 'w'), indent=2, default=str); log('Analysis saved.')

## 9. Plot: membership-AUC gap to Retrain-1 (same precision) vs bit-width

In [ ]:
import matplotlib.pyplot as plt
if QD.empty: print('Nothing to plot (NO_FP32_GATE_PASSED).')
for ds in (CFG['datasets'] if not QD.empty else []):
    for dose in TO_QUANTIZE[ds]:
        ft = f'FT dose={dose}'; x = QD[QD.dataset == ds]
        fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=True)
        for ax, (scheme, mode) in zip(axes, itertools.product(CFG['schemes'], CFG['scale_modes'])):
            for model, cond, style in [('Original', 'quantized', 'k--'), (ft, 'quantized', 'C0-o'), (ft, 'matched_noise', 'C1-s'),
                                       ('Retrain-2', 'quantized', 'C2:^')]:
                ys, es = [], []
                for b in [32] + CFG['bits']:
                    sel = x[(x.condition == 'fp32')] if b == 32 else x[(x.bits == b) & (x.scheme == scheme) & (x.scale_mode == mode) & (x.condition == cond)]
                    a = sel.pivot_table(index='seed', columns='model', values='auc'); gp = a[model] - a['Retrain-1']
                    ys.append(gp.mean()); es.append(gp.std())
                ax.errorbar(range(len(ys)), ys, yerr=es, fmt=style, capsize=3, label=f'{model} ({cond})' if model != 'Original' else 'Original')
            ax.axhspan(-GATE[ds]['band_fp32'], GATE[ds]['band_fp32'], color='grey', alpha=0.2, label='FP32 band')
            ax.set_xticks(range(1 + len(CFG['bits']))); ax.set_xticklabels(['FP32'] + [f'INT{b}' for b in CFG['bits']])
            ax.set_title(f'{ds} dose {dose}: {scheme}, {mode}'); ax.axhline(0, c='k', lw=0.5)
        axes[0].set_ylabel('AUC(model) - AUC(Retrain-1)'); axes[0].legend(fontsize=7)
        plt.tight_layout(); plt.savefig(f'{OUT_DIR}/fig_gap_{ds}_dose{dose}.png', dpi=160); plt.show()

## 10. Export everything as a ZIP

In [ ]:
def export():
    base = os.path.basename(OUT_DIR.rstrip('/'))
    shutil.make_archive(base, 'zip', OUT_DIR)
    log(f'ZIP created: {base}.zip ({os.path.getsize(base + ".zip")/1e6:.1f} MB)')
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        from IPython.display import FileLink, display
        print('Kaggle: click the link below, or use the Output panel (/kaggle/working). In a committed version the ZIP stays in the Output tab.')
        display(FileLink(f'{base}.zip')); return
    try:
        from google.colab import files; files.download(f'{base}.zip')
    except Exception:
        print('Download the ZIP from the file panel on the left.')
export()

### Emergency export
If the session is about to disconnect, run only the cell below. It zips whatever has been saved so far.

In [ ]:
export()